### Agentic RAG System 

In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [52]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain.agents import create_agent
from langchain_core.tools import tool


In [53]:
loader = PyPDFLoader("../Data/medical_report.pdf")
data = loader.load()
len(data)


9

In [55]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_chunk = splitter.split_documents(data)
len(split_chunk)

26

In [56]:
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = Chroma.from_documents(
    documents=split_chunk,
    embedding=embedding
)

In [ ]:
# res = vector_store.similarity_search("patient name")
# res[0].page_content

'MD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\nDr Kiran Bhargava Pathak\nMD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\n*474764803*\n.\nPage 7 of 9'

### Agent - tool | llm | system_prompt

In [95]:
@tool                         
def get_tool(query:str):
    """
    This tool can help you to retrieve the relevant data of the PDF Documents, and these documents have detail about medical report.
    
    """
    
    print("tool called :", query)
    docs = vector_store.similarity_search(query=query, k=4)
    context = ""
    
    for doc in docs:
        context = doc.page_content + "/n"
   
    return context
   

In [96]:
print(get_tool.invoke("doctor name"))

tool called : doctor name
Report Status    
Female
27 Years:
:
:
:
Age
Gender
Reported        
P
9/7/2025   4:56:00PM
DR NITIN NAHAR
474764803
Ms. NIKITA  CHUDHARY:
:
:
:
:
Name        
Lab No.    
Ref By 
Collected       
A/c Status 
10/7/2025  6:31:50PM
Final
LPL - Bhopal Lab II
Plot No.05, Mandakini Housing Society, 
Near Apurti Shopping Mall, Kolar Main 
Road, Bhopal, M.P. -  462042
::Collected at            Processed at             BHOPAL CC-82
Mr Rachel V John Pata So Vitus John Mig 26 
Graund,Indrapuri, Phone: 8770817968
 
Test Report      
Test Name Results Units Bio. Ref. Interval
Decreased Levels
· Inadequate exposure to sunlight
· Dietary deficiency
· Vitamin D malabsorption
· Severe  Hepatocellular disease
· Drugs like Anticonvulsants
· Nephrotic syndrome
Increased levels
Vitamin D intoxication 
Dr.Swapnil Gupta
MD, Pathology
Chief of Laboratory                            
Dr Lal PathLabs Ltd
Dr Kiran Bhargava Pathak
MD, Pathology
Chief of Laboratory                        

In [97]:
llm = ChatOllama(model="gemma4:31b-cloud")

In [100]:
system_prompt= """
            You are a helpul assistant that answer questions using retrieved context,
            ALWAYS use the 'get_tool' tool for question requiring external resources.
            """

In [103]:
agent = create_agent(
    model=llm,
    tools=[get_tool],
    system_prompt=system_prompt
)

In [112]:
query = "What is doctor name and patient name and address ?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

res= response["messages"][-1].content

print(res)

tool called : doctor name, patient name, and patient address
tool called : patient name, doctor name, and address from the medical report
Based on the medical report:

*   **Patient Name:** Ms. Nikita Chudhary
*   **Doctor Name (Referring Doctor):** Dr. Nitin Nahar
*   **Patient Address:** Mr Rachel V John Pata So Vitus John Mig 26, Graund, Indrapuri (Phone: 8770817968)
